## ML

In [95]:
!pip install optuna


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [139]:
# Imports
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance
import lightgbm as lgb
import optuna
from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance

In [143]:
# Check for existence of credentials, if not found, fail early
load_dotenv(override=True)

def require_env(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise EnvironmentError(f"Environment variable {key} is missing")
    return val

In [144]:
# Database credentials
host = "localhost"
dbname = require_env("DB_NAME")
user = require_env("DB_USER")
password = require_env("DB_PASSWORD")

### Import data

In [4]:
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:5432/{dbname}", isolation_level="AUTOCOMMIT")

df = pd.read_sql(
    "SELECT * FROM public.ml_monthly_features ORDER BY listings_id, target_month",
    engine,
    parse_dates=["target_month", "feature_cohort_date"],
    dtype_backend="numpy_nullable" # avoid postgres to python/pandas dtype conversion problems
)

### Health checks

In [145]:
print(df.shape)
print(df["new_reviews"].describe())
print((df["new_reviews"] == 0).mean())  # share of zero months

(137274, 50)
count    137274.0
mean     0.938714
std      2.424751
min           0.0
25%           0.0
50%           0.0
75%           1.0
max         160.0
Name: new_reviews, dtype: Float64
0.6953465332109504


In [146]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 137274 entries, 0 to 137273
Data columns (total 50 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   month_key                     137274 non-null  string        
 1   listings_id                   137274 non-null  Int64         
 2   target_month                  137274 non-null  datetime64[us]
 3   feature_cohort_date           137274 non-null  datetime64[us]
 4   feature_lag_days              137274 non-null  Int64         
 5   new_reviews                   137274 non-null  Int64         
 6   room_type                     137274 non-null  category      
 7   building_type                 137274 non-null  category      
 8   accommodates                  137274 non-null  Int64         
 9   bathrooms                     136132 non-null  Float64       
 10  bedrooms                      116013 non-null  Int64         
 11  beds                    

In [147]:
print(df.describe())

                 listings_id                target_month  \
count               137274.0                      137274   
mean    552641030706074112.0  2026-02-25 18:23:45.272083   
min                   3176.0         2025-10-01 00:00:00   
25%               22500999.0         2025-12-01 00:00:00   
50%               53288374.0         2026-03-01 00:00:00   
75%    1131130081651908864.0         2026-06-01 00:00:00   
max    1715912659116099328.0         2026-08-01 00:00:00   
std     600367723075171328.0                         NaN   

              feature_cohort_date  feature_lag_days  new_reviews  \
count                      137274          137274.0     137274.0   
mean   2026-01-09 04:49:55.935136         47.565154     0.938714   
min           2025-09-23 00:00:00               4.0          0.0   
25%           2025-09-23 00:00:00               8.0          0.0   
50%           2025-12-27 00:00:00              36.0          0.0   
75%           2026-03-28 00:00:00              69.0

In [148]:
print(df.nlargest(10, "new_reviews")[
    ["listings_id", "target_month", "new_reviews", "reviews_before", "accommodates"]
])

                listings_id target_month  new_reviews  reviews_before  \
77953    721527807549293451   2026-08-01          160            1733   
77952    721527807549293451   2026-07-01          143            1733   
127295  1461455135108407782   2026-08-01          113             116   
77951    721527807549293451   2026-06-01          111            1604   
123294  1414395977088158733   2026-08-01          110             624   
131044  1508712027277107910   2026-01-01          101              70   
123292  1414395977088158733   2026-06-01           89             624   
123290  1414395977088158733   2026-04-01           88             624   
118906  1359377017583012847   2026-08-01           83             496   
123293  1414395977088158733   2026-07-01           74             624   

        accommodates  
77953              2  
77952              2  
127295             2  
77951              2  
123294             2  
131044             2  
123292             2  
123290      

In [149]:
print(df.columns.tolist())
print(df["feature_lag_days"].describe())
print(df.groupby("listings_id").size().describe())


['month_key', 'listings_id', 'target_month', 'feature_cohort_date', 'feature_lag_days', 'new_reviews', 'room_type', 'building_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'neighbourhood_group_cleansed', 'neighbourhood_cleansed', 'price', 'price_is_missing', 'price_per_person', 'price_source', 'minimum_nights', 'stay_type', 'has_license', 'availability_30', 'availability_365', 'has_availability', 'reviews_before', 'rating_before', 'cleanliness_before', 'location_before', 'host_is_superhost', 'host_listings_count', 'month_of_year', 'amenities_count', 'has_wifi', 'has_kitchen', 'has_washer', 'has_air_conditioning', 'has_free_parking', 'has_elevator', 'allows_pets', 'has_tv', 'has_self_check_in', 'has_patio_or_balcony', 'has_shared_bathroom', 'is_reappearance', 'cohorts_since_previous', 'days_since_first_seen', 'new_reviews_lag_1', 'new_reviews_lag_2', 'new_reviews_lag_3', 'new_reviews_avg_prev_3']
count     137274.0
mean     47.565154
std      41.779817
min            4.0
25%  

In [150]:
print(df.groupby("feature_cohort_date")["listings_id"].nunique())
print(df[df["feature_lag_days"] > 120]["target_month"].value_counts().sort_index())

feature_cohort_date
2025-09-23    13530
2025-12-27    12043
2026-03-28     5424
2026-06-26    11693
Name: listings_id, dtype: int64
target_month
2026-02-01     111
2026-03-01     111
2026-04-01      76
2026-05-01    6754
2026-06-01    6754
2026-07-01      97
2026-08-01     151
Name: count, dtype: int64


In [151]:
missing_fractions = df.isna().mean()
missing_fractions.sort_values(ascending=False)

price                           0.693205
price_source                    0.693205
price_per_person                0.693205
beds                            0.352303
cohorts_since_previous          0.342272
new_reviews_lag_3               0.329604
new_reviews_lag_2               0.222708
location_before                 0.210491
rating_before                   0.210309
cleanliness_before              0.210309
bedrooms                        0.154880
new_reviews_lag_1               0.111354
new_reviews_avg_prev_3          0.111354
has_availability                0.030975
bathrooms                       0.008319
host_is_superhost               0.004145
has_shared_bathroom             0.001056
host_listings_count             0.001027
minimum_nights                  0.000554
stay_type                       0.000554
neighbourhood_group_cleansed    0.000000
feature_cohort_date             0.000000
price_is_missing                0.000000
neighbourhood_cleansed          0.000000
building_type   

### Preprocessing

In [152]:
target = "new_reviews"

id_cols = ["month_key", "listings_id", "target_month", "feature_cohort_date"]

categorical = [
    "room_type",
    "building_type",
    "stay_type",
    "price_source",
    "neighbourhood_group_cleansed",
    "neighbourhood_cleansed",
]

In [153]:
feature_cols = [c for c in df.columns if c not in id_cols + [target]]

# drop columns that are empty in every row
empty = [c for c in feature_cols if df[c].isna().all()]
feature_cols = [c for c in feature_cols if c not in empty]
categorical = [c for c in categorical if c in feature_cols]

In [154]:
print("empty:", empty)
print(len(feature_cols), "features,", len(categorical), "of them categorical")

empty: []
45 features, 6 of them categorical


In [155]:
# helper function
def to_lgbm(data):
    """Nullable pandas dtypes -> what LightGBM accepts. NaN stays NaN."""
    out = data[feature_cols].copy()
    for c in feature_cols:
        if c in categorical:
            out[c] = out[c].astype("category")
        else:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("float64")
    return out

In [156]:
months = np.sort(df["target_month"].unique())
print(months)

['2025-10-01T00:00:00.000000' '2025-11-01T00:00:00.000000'
 '2025-12-01T00:00:00.000000' '2026-01-01T00:00:00.000000'
 '2026-02-01T00:00:00.000000' '2026-03-01T00:00:00.000000'
 '2026-04-01T00:00:00.000000' '2026-05-01T00:00:00.000000'
 '2026-06-01T00:00:00.000000' '2026-07-01T00:00:00.000000'
 '2026-08-01T00:00:00.000000']


In [157]:
cohorts = np.sort(df["feature_cohort_date"].unique())

In [158]:
# Train-Val-Test-Split
test  = df[df["feature_cohort_date"] == cohorts[-1]]
val   = df[df["feature_cohort_date"] == cohorts[-2]]
train = df[df["feature_cohort_date"] <  cohorts[-2]]

In [164]:

X_train, y_train = to_lgbm(train), train[target].astype("float64")
X_val, y_val = to_lgbm(val),   val[target].astype("float64")

print(f"train len: {len(train)}  val len: {len(val)}  test len: {len(test)}")

train len: 97508  val len: 16380  test len: 23386


In [165]:
for column in feature_cols:
    tr = train[column].isna().mean()
    te = test[column].isna().mean()
    if abs(tr - te) > 0.2:
        print(f"{column} train={tr} test={te}")

price train=0.7280530828239734 test=0.3330197554092192
price_per_person train=0.7280530828239734 test=0.3330197554092192
price_source train=0.7280530828239734 test=0.3330197554092192
cohorts_since_previous train=0.4577880789268573 test=0.0523390062430514
new_reviews_lag_2 train=0.29341182261968246 test=0.0523390062430514
new_reviews_lag_3 train=0.44011773392952375 test=0.0523390062430514


### Baseline Metrics

In [ ]:
# Test different metrics
def score(y_true, pred):
    pred = np.clip(np.asarray(pred, dtype=float), 0.05, None)
    return {
        "MeanAbsoluteError": round(mean_absolute_error(y_true, pred), 4),
        "PoissonDeviance": round(mean_poisson_deviance(y_true, pred), 4),
        "Spearman": round(spearmanr(y_true, pred).correlation, 4),
    }

In [178]:
baselines = {
    "always_zero": np.zeros(len(val)),
    "train_mean":  np.full(len(val), y_train.mean())
}

for name, pred in baselines.items():
    print(f"{name:12s}", score(y_val, pred))

always_zero  {'MAE': 2.2599, 'PoissonDeviance': 16.3421, 'Spearman': np.float64(nan)}
train_mean   {'MAE': 2.0601, 'PoissonDeviance': 5.7901, 'Spearman': np.float64(nan)}


/tmp/ipykernel_1086490/3911289480.py:6: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "Spearman": round(spearmanr(y_true, pred).correlation, 4),
/tmp/ipykernel_1086490/3911289480.py:6: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "Spearman": round(spearmanr(y_true, pred).correlation, 4),


### Model Training

Here the target is a non-negative counter variable with many nulls, therefore Poisson is chosen as the training objective.

In [168]:
params = dict(
    objective="poisson",   # count data with many zeros
    metric="poisson",
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=100,
    verbosity=-1,
)

dtrain = lgb.Dataset(
    X_train, y_train,
    categorical_feature=categorical,
    params={"feature_pre_filter": False},
    free_raw_data=False,
)
dval = lgb.Dataset(X_val, y_val, reference=dtrain, free_raw_data=False)

In [169]:
# Train model 
model = lgb.train(
    params, dtrain,
    num_boost_round=2000,
    valid_sets=[dval],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)

/data/jakob/Desktop/data_engineering/c_herumspielen/aa_real_dbt_10.-12.26/.venv/lib/python3.12/site-packages/lightgbm/basic.py:2515: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning("Overriding the parameters from Reference Dataset.")


Training until validation scores don't improve for 100 rounds
[200]	valid_0's poisson: -0.555482
Early stopping, best iteration is:
[157]	valid_0's poisson: -0.558644


In [170]:
pred = model.predict(X_val, num_iteration=model.best_iteration)
print("LightGBM  ", score(y_val, pred))
print("best_iteration:", model.best_iteration)

LightGBM   {'MAE': 1.1935, 'PoissonDeviance': 1.4875, 'Spearman': np.float64(0.742)}
best_iteration: 157


### Model basic HP tuning

In [171]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    p = dict(
        objective="poisson",
        metric="poisson",
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 20, 500, log=True),
        feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),
        bagging_freq=1, # required, otherwise bagging_fraction is ignored
        lambda_l2=trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        verbosity=-1,
    )

    m = lgb.train(
        p, dtrain,
        num_boost_round=2000,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    trial.set_user_attr("best_iteration", m.best_iteration)

    pred = m.predict(X_val, num_iteration=m.best_iteration)
    return mean_poisson_deviance(y_val, np.clip(pred, 0.05, None))

In [172]:
# Define study for optimization
study = optuna.create_study(
    study_name="new_reviews",
    storage="sqlite:///optuna_reviews.db",  # resumes on rerun
    direction="minimize",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=42),
)

In [173]:
study.optimize(objective, n_trials=50, show_progress_bar=True)

Best trial: 17. Best value: 0.607541: 100%|██████████| 50/50 [13:49<00:00, 16.59s/it]


In [174]:
print(len(study.trials), "best MAE:", round(study.best_value, 4), "best params:", study.best_params)

71 best MAE: 0.6075 best params: {'learning_rate': 0.013853772275778184, 'num_leaves': 217, 'min_data_in_leaf': 483, 'feature_fraction': 0.7839133831004486, 'bagging_fraction': 0.7307480029585662, 'lambda_l2': 8.622960906579019}


In [183]:
score(y_val, best_model.predict(X_val))["MAE"]

1.1882

In [184]:
best_new = {score(y_val, best_model.predict(X_val))["MAE"] : study.best_params }
pd.Series(best_new).to_json("best_hyperparameter_configuration.json")

In [175]:
best_params = {**study.best_params, "objective": "poisson", "metric": "poisson", "bagging_freq": 1, "verbosity": -1}
n_rounds = study.best_trial.user_attrs["best_iteration"]

best_model = lgb.train(best_params, dtrain, num_boost_round=n_rounds)
best_model.save_model("model_new_reviews.txt")

### Inspecting results

In [176]:
print("tuned", score(y_val, best_model.predict(X_val)))
print("last_month", score(y_val, val["new_reviews_lag_1"].fillna(0)))

tuned {'MAE': 1.1882, 'PoissonDeviance': 1.4896, 'Spearman': np.float64(0.744)}
last_month {'MAE': 1.2728, 'PoissonDeviance': 2.5121, 'Spearman': np.float64(0.6841)}


In [177]:
pd.Series(best_model.feature_importance("gain"), index=best_model.feature_name()).sort_values(ascending=False).head(15)

new_reviews_avg_prev_3    2.339680e+06
new_reviews_lag_1         8.433179e+05
minimum_nights            6.161403e+05
reviews_before            1.873587e+05
neighbourhood_cleansed    1.728719e+05
stay_type                 1.491115e+05
availability_30           9.346383e+04
availability_365          7.490323e+04
new_reviews_lag_2         5.601864e+04
days_since_first_seen     4.125386e+04
price_per_person          4.063971e+04
feature_lag_days          2.827554e+04
new_reviews_lag_3         2.510224e+04
host_is_superhost         2.396923e+04
month_of_year             2.092645e+04
dtype: float64

In [ ]:
# mini class for permutation importance
class LGBWrapper:
    def __init__(self, booster):
        self.booster = booster
    def fit(self, X, y):
        return self
    def predict(self, X):
        return self.booster.predict(X)
    def score(self, X, y):
        return -mean_poisson_deviance(y, np.clip(self.predict(X), 0.05, None))

In [ ]:
perm = permutation_importance(LGBWrapper(best_model), X_val, y_val, n_repeats=5, random_state=42, n_jobs=-1,)

new_reviews_avg_prev_3    1.134934
minimum_nights            0.323140
new_reviews_lag_1         0.296537
reviews_before            0.121028
availability_30           0.106435
availability_365          0.099959
neighbourhood_cleansed    0.041838
stay_type                 0.034467
new_reviews_lag_2         0.025745
beds                      0.013396
rating_before             0.013333
has_self_check_in         0.012833
host_is_superhost         0.012205
amenities_count           0.009393
new_reviews_lag_3         0.006704
dtype: float64

In [ ]:
pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False).head(15)

new_reviews_avg_prev_3    1.134934
minimum_nights            0.323140
new_reviews_lag_1         0.296537
reviews_before            0.121028
availability_30           0.106435
availability_365          0.099959
neighbourhood_cleansed    0.041838
stay_type                 0.034467
new_reviews_lag_2         0.025745
beds                      0.013396
rating_before             0.013333
has_self_check_in         0.012833
host_is_superhost         0.012205
amenities_count           0.009393
new_reviews_lag_3         0.006704
dtype: float64

TODO:
* feature engineering: review previous review activity features 
* convert to pipeline
* integrate with Airflow